# 5. ONNX Runtime inference

Run the exported graphs with ONNX Runtime on CPU. Install the ONNX extra before opening this notebook. CUDA providers can be supplied explicitly when using a compatible ONNX Runtime GPU installation.

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


In [ ]:
from PIL import Image
from parseq_ocr.inference import ONNXRecognizer
from parseq_ocr.data import scan_pairs

recognizer = ONNXRecognizer(config["output_dir"] / "deployment")
records = scan_pairs(config["dataset_root"] / "val", config["normalization"])[:3]
images = []
for row in records:
    with Image.open(row["path"]) as image:
        images.append(image.convert("RGB"))
predictions = recognizer.infer(images)
for row, prediction in zip(records, predictions):
    print({"ground_truth": row["label"], "prediction": prediction})

Standard split graphs retain encoder memory between decoder calls. Fast Tiny runs its complete encoder/prediction/refinement graph in one ONNX Runtime call. This CPU example demonstrates portability, not the GPU speed reported for TensorRT. Use notebook 7 for complete accuracy and fresh-process benchmarks.